#Fine-tune GPT or GPT-2 for creative story generation.

Step 1: Install Dependencies

In [2]:
!pip install -q transformers datasets torch accelerate

Step 2: Load the Model and Tokenizer

In [3]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

# Load the tokenizer and model
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a pad token by default, so we set it to the End-Of-String (EOS) token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("Model and Tokenizer loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and Tokenizer loaded successfully!


Step 3: Load and Preprocess the Dataset

In [4]:
from datasets import load_dataset

# Load a small subset of the TinyStories dataset for fast fine-tuning
print("Loading dataset...")
dataset = load_dataset("roneneldan/TinyStories", split="train[:5000]")

# Tokenization function
def tokenize_function(examples):
    # Tokenize the text, truncate to 128 tokens for speed and memory efficiency
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# For language modeling, the labels are the exact same as the input_ids
# The model internally shifts the labels to predict the *next* token
def copy_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

print("Formatting for PyTorch...")
tokenized_datasets = tokenized_datasets.map(copy_labels, batched=True)
tokenized_datasets.set_format("torch")

print("Dataset ready!")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

...

Tokenizing dataset...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Formatting for PyTorch...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset ready!


Step 4: Configure and Run the Trainer

In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-story-generator",
    num_train_epochs=3,              # 3 passes through the dataset
    per_device_train_batch_size=8,   # Batch size per GPU
    save_steps=2000,                 # Save checkpoint frequency
    logging_steps=100,               # Print loss every 100 steps
    learning_rate=5e-5,              # Standard learning rate for fine-tuning
    fp16=True,                       # Use mixed precision for faster training on GPU
    report_to="none"                 # Disable wandb/tensorboard logging for simplicity
)

# The data collator dynamically pads the batches during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # mlm=False tells the collator we are doing Causal LM (GPT-2 style), not Masked LM (BERT style)
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training complete!")

# Save the final fine-tuned model and tokenizer
trainer.save_model("./gpt2-story-generator-final")
tokenizer.save_pretrained("./gpt2-story-generator-final")

Starting training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.217763
200,2.035364
300,2.009177
400,1.978296
500,1.947224
600,1.913480
700,1.832131
800,1.814234
900,1.806320
1000,1.792730


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-story-generator-final/tokenizer_config.json',
 './gpt2-story-generator-final/tokenizer.json')

Step 5: Generate a Creative Story

In [6]:
from transformers import pipeline

# Create a text generation pipeline using our newly fine-tuned model
story_generator = pipeline(
    "text-generation",
    model="./gpt2-story-generator-final",
    tokenizer="./gpt2-story-generator-final",
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available
)

# The starting seed for the story
prompt = "Once upon a time in a dark, magical forest,"

print("Generating story...\n" + "-"*50)

# Generate the text
generated_output = story_generator(
    prompt,
    max_length=150,           # Maximum length of the story
    num_return_sequences=1,   # Generate 1 story
    temperature=0.8,          # Higher temp = more creative/random, lower = more predictable
    top_p=0.9,                # Nucleus sampling for better coherence
    repetition_penalty=1.2,   # Penalize repeating the same words
    do_sample=True            # Required when using temperature/top_p
)

print(generated_output[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'do_sample', 'num_return_sequences', 'top_p', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating story...
--------------------------------------------------
Once upon a time in a dark, magical forest, there lived an elderly fairy. She was very kind and always helped people who needed it most.
One day, the old lady found something she had never seen before on the ground. It was tiny stones! The old lady carefully picked up the stones so that they could be used to make things from them. Her little helpers were delighted with their work.  The old woman gave them some candy bars for each of her children and grandchildren. After making delicious cookies and playing games, the young adults returned to help out at the fair. They played together all day long and made lots of tasty treats - including cakes, pies & cake mixes! Everyone loved helping this beautiful person every single day. From then until today when it started raining again, the wise old lady's kindness increased as well! Now everyone wanted to see more wonderful sights too, especially if they came back soon after

Testing

In [7]:
# A list of different starting seeds
prompts = [
    "Once upon a time in a dark, magical forest,",
    "The little brown dog dug a hole in the yard and found",
    "On a rainy Tuesday, Mia looked out the window and saw a flying"
]

for prompt in prompts:
    print(f"\n✨ Prompt: {prompt}")
    print("-" * 50)

    generated_output = story_generator(
        prompt,
        max_length=150,
        num_return_sequences=1,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True
    )

    print(generated_output[0]["generated_text"])

Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ Prompt: Once upon a time in a dark, magical forest,
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Once upon a time in a dark, magical forest, there lived an elf named Bob. He was very brave and always wanted to do something special - so he went on adventures!
One day, Bob decided he would go explore the world around him by himself. So he started looking for his way out of the woods. 
So when he arrived at a big clearing near the edge of the jungle area with no sign that anyone had left, he made it back inside safely. But then suddenly he saw strange things floating through the air all too quickly! As soon as they touched Bob's face, he froze up and screamed into his ears. The mysterious creatures were making noises like laughing or shouting 'hello'. 
Bob knew what happened next because even though nobody seemed to know anything about them, from where did their presence come from? Suddenly he heard some words coming down from above: "I'm here!", but everyone didn't listen anymore. They just stayed silent until finally one more thing appeared behind them - this time using its wings t

Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The little brown dog dug a hole in the yard and found some rocks. He looked around, but he couldn't find anything to play with. Then he saw an old man walking by who was very friendly.

"What's wrong, Timmy?" asked the elderly gentleman. "I donâ€™t know what you are doing." ÂœTimmy? What do we need help with?" 
He stepped closer and pointed at a big pile of stones on the ground. The other two dogs ran over to him and said, â¢Thank You! I can pick them up for us!"    It took longer than they thought so Tim felt sorry for the stranger. But then the wise owl appeared from behind a tree nearby and offered to supply his wagon with food if they could get one together.
When they got there it seemed like everyone had been careful not come near the stone piles until their turn. They quickly packed the boxes onto the back of my truck and drove away as fast breezes rolled through the air. After that, Timmy put down the rock and thanked the birdie birds nestling under the trees. When they were gon